# Lab 02 — Medical Imaging with Point / Photometric Transforms (Tasks 1 – 3 + Extra)
One code cell per task. Run **Setup** first and set the file paths in `PATHS`
(relative paths, as the submission guidelines require). Missing files fall back to
synthetic data so every cell runs. The **Extra** cell covers piecewise-linear
transforms, which the exam announcement names but the task sheet does not.

In [ ]:
# ===================== SETUP (run first) =====================
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

PATHS = {
    "xray": "data/sample_xray.png",     # Task 1 (Kaggle COVID/Pneumonia/Normal chest X-ray)
    "ct": "data/ct.png",                # Task 2 (Heart CT & MRI dataset, matched pair)
    "mri": "data/mri.png",
    "echo": "data/echo.mp4",            # Task 3 (EchoNet-Dynamic clip)
}
os.makedirs("output", exist_ok=True)
SHOW_LIVE = False      # True -> cv2.imshow window (local Python only). False -> works in Jupyter.


def show(imgs, titles, cols=4, cmap="gray"):
    rows = (len(imgs) + cols - 1) // cols
    plt.figure(figsize=(4.5 * cols, 4 * rows))
    for i, (im, t) in enumerate(zip(imgs, titles)):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap=cmap)
        plt.title(t, fontsize=10); plt.axis("off")
    plt.tight_layout(); plt.show()


# ---------- point transforms used by all three tasks ----------
def log_transform(g):
    """s = c*log(1+r), c = 255/log(1+r_max). Expands dark values."""
    f = g.astype(np.float64)                    # float first: avoid uint8 overflow
    c = 255 / np.log(1 + max(f.max(), 1))
    return np.uint8(np.clip(c * np.log1p(f), 0, 255))


def gamma_transform(img, gamma):
    """s = 255*(r/255)^gamma via a lookup table. gamma<1 brightens, >1 darkens."""
    table = np.clip(255 * (np.arange(256) / 255.0) ** gamma, 0, 255).astype(np.uint8)
    return cv2.LUT(img, table)                  # works on gray and colour


def gray_world(bgr):
    """Colour balance: scale each channel so its mean equals the overall mean."""
    f = bgr.astype(np.float64)
    avg = f.reshape(-1, 3).mean(axis=0)
    f *= avg.mean() / np.maximum(avg, 1e-6)
    return np.uint8(np.clip(f, 0, 255))


# ---------- synthetic fallbacks ----------
def synthetic_xray():
    rng = np.random.default_rng(1)
    img = np.full((512, 512), 18.0)
    cv2.ellipse(img, (256, 270), (200, 230), 0, 0, 360, 40, -1)          # body
    for cx in (170, 342):
        cv2.ellipse(img, (cx, 250), (75, 150), 0, 0, 360, 25, -1)        # lungs (dark)
    for y in range(120, 420, 34):
        cv2.ellipse(img, (256, y), (190, 40), 0, 200, 340, 70, 7)        # ribs
    cv2.rectangle(img, (240, 60), (272, 460), 80, -1)                    # spine
    cv2.circle(img, (330, 330), 28, 52, -1)                              # "fluid" patch
    img = cv2.GaussianBlur(img, (9, 9), 0) + rng.normal(0, 2, img.shape)
    return np.clip(img, 0, 255).astype(np.uint8)                         # underexposed: max ~85


def synthetic_ct_mri():
    ct = np.full((400, 400), 20, np.uint8)
    cv2.ellipse(ct, (200, 200), (150, 120), 0, 0, 360, 90, -1)
    cv2.ellipse(ct, (200, 200), (150, 120), 0, 0, 360, 230, 6)           # sharp boundary
    cv2.circle(ct, (170, 190), 45, 140, 3); cv2.circle(ct, (240, 210), 35, 140, 3)
    mri = np.full((400, 400), 30.0)
    cv2.ellipse(mri, (200, 200), (148, 118), 0, 0, 360, 110, -1)
    cv2.circle(mri, (170, 190), 45, 180, -1); cv2.circle(mri, (240, 210), 35, 60, -1)  # soft tissue
    mri = cv2.GaussianBlur(mri, (31, 31), 0)                             # blurry edges
    return ct, np.clip(mri, 0, 255).astype(np.uint8)


def synthetic_echo(path, n=60):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    rng = np.random.default_rng(3)
    out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), 20, (320, 240))
    for i in range(n):
        f = np.zeros((240, 320), np.float64)
        cv2.ellipse(f, (160, 0), (220, 230), 90, -35, 35, 35, -1)        # ultrasound fan
        r = int(40 + 12 * np.sin(2 * np.pi * i / 20))                    # beating chamber
        cv2.ellipse(f, (160, 130), (r, int(r * 1.4)), 0, 0, 360, 10, -1)
        cv2.ellipse(f, (160, 130), (r + 8, int(r * 1.4) + 8), 0, 0, 360, 70, 4)
        f += rng.normal(0, 10, f.shape) * (f > 0)                         # speckle
        f[rng.random(f.shape) > 0.997] = 255                              # backscatter
        out.write(cv2.cvtColor(np.clip(f, 0, 255).astype(np.uint8), cv2.COLOR_GRAY2BGR))
    out.release()


print("Setup done | OpenCV", cv2.__version__)

In [ ]:
# ===================== TASK 1 — Diagnostic Enhancement of Chest X-Rays =====================
xray = cv2.imread(PATHS["xray"], cv2.IMREAD_GRAYSCALE)
if xray is None:
    print("X-ray not found -> synthetic underexposed X-ray")
    xray = synthetic_xray()
print(f"Raw X-ray: shape={xray.shape}, range=[{xray.min()}, {xray.max()}], mean={xray.mean():.1f}")

# 1) Contrast enhancement — histogram equalization (s_k = 255 * CDF(r_k))
eq = cv2.equalizeHist(xray)

# 2) Colour mapping — false colour (blue = low, red = high intensity)
heat = cv2.applyColorMap(eq, cv2.COLORMAP_JET)

# 3) Colour balance — gray-world correction removes the colour cast of the heatmap
balanced = gray_world(heat)

# 4) Colour filtering — keep only the densest tissue (highest intensities)
T_DENSE = 200
_, dense_mask = cv2.threshold(eq, T_DENSE, 255, cv2.THRESH_BINARY)
dense_only = cv2.bitwise_and(eq, eq, mask=dense_mask)

# 5) Log transform on the RAW underexposed image — expands dark background (rib edges)
log_img = log_transform(xray)

# 6) Power-law with gamma < 1 — lifts midtones, softens harsh bone/lung contrast
gamma_img = gamma_transform(eq, 0.6)

show([xray, eq, heat, balanced, dense_only, log_img, gamma_img],
     ["Raw X-ray", "Histogram equalized", "COLORMAP_JET", "Colour balanced (gray world)",
      f"Dense tissue only (T={T_DENSE})", "Log transform (raw)", "Gamma 0.6 (on equalized)"])

plt.figure(figsize=(12, 3.5))
for i, (im, t) in enumerate([(xray, "Raw"), (eq, "Equalized"), (log_img, "Log"), (gamma_img, "Gamma 0.6")]):
    plt.subplot(1, 4, i + 1); plt.hist(im.ravel(), 256, range=(0, 256), color="gray"); plt.title(t + " histogram")
plt.tight_layout(); plt.show()

for name, im in [("equalized", eq), ("heatmap", heat), ("balanced", balanced),
                 ("dense", dense_only), ("log", log_img), ("gamma", gamma_img)]:
    cv2.imwrite(f"output/task1_{name}.png", im)
print("""Notes:
- Equalization spreads the narrow, dark histogram over 0-255, revealing lung structure.
- The eye separates colours far better than ~30 gray shades, so JET makes subtle
  intensity boundaries (e.g. fluid) visible. It adds no new information.
- Log expands the darkest values most, so faint rib edges in the background appear.
- Gamma < 1 brightens midtones, lowering the bone-vs-soft-tissue contrast.""")

In [ ]:
# ===================== TASK 2 — Multi-Modal Cardiac Image Fusion =====================
ct = cv2.imread(PATHS["ct"], cv2.IMREAD_GRAYSCALE)
mri = cv2.imread(PATHS["mri"], cv2.IMREAD_GRAYSCALE)
if ct is None or mri is None:
    print("CT/MRI pair not found -> synthetic matched pair")
    ct, mri = synthetic_ct_mri()
mri = cv2.resize(mri, (ct.shape[1], ct.shape[0]))          # sizes must match for blending

# 1) Equalize each modality independently
ct_eq, mri_eq = cv2.equalizeHist(ct), cv2.equalizeHist(mri)

# 2) Colour mapping: CT kept gray (structure), MRI mapped to a heatmap (soft tissue)
ct_col = cv2.cvtColor(ct_eq, cv2.COLOR_GRAY2BGR)           # 3 channels to blend with MRI
mri_col = cv2.applyColorMap(mri_eq, cv2.COLORMAP_JET)
overlay = cv2.addWeighted(ct_col, 0.5, mri_col, 0.5, 0)    # simple equal overlay

# 3) Weighted fusion: heavier CT weight keeps sharp anatomical edges
ALPHA, BETA = 0.7, 0.3
fused = cv2.addWeighted(ct_col, ALPHA, mri_col, BETA, 0)   # ALPHA*CT + BETA*MRI + 0
weights = [(0.3, 0.7), (0.5, 0.5), (0.7, 0.3), (0.9, 0.1)]
sweep = [cv2.addWeighted(ct_col, a, mri_col, b, 0) for a, b in weights]

# 4) Log then power-law on the fused result: lift crushed darks, tame highlights
fused_gray = cv2.cvtColor(fused, cv2.COLOR_BGR2GRAY)
final_gray = gamma_transform(log_transform(fused_gray), 1.5)
final_color = gamma_transform(fused, 1.2)                  # colour version (LUT per channel)

clip_before = [(fused_gray == 0).mean() * 100, (fused_gray == 255).mean() * 100]
clip_after = [(final_gray == 0).mean() * 100, (final_gray == 255).mean() * 100]
print(f"% pure black/white before: {clip_before[0]:.2f}/{clip_before[1]:.2f}  "
      f"after log+gamma: {clip_after[0]:.2f}/{clip_after[1]:.2f}")

# 5) Comparative analysis
show([ct, mri, overlay, fused, final_gray, final_color],
     ["CT (raw)", "MRI (raw)", "Overlay 0.5/0.5", f"Fused CT {ALPHA} / MRI {BETA}",
      "Fused -> log -> gamma 1.5", "Fused colour, gamma 1.2"], cols=3)
show(sweep, [f"CT {a} / MRI {b}" for a, b in weights])
cv2.imwrite("output/task2_fused.png", fused)
cv2.imwrite("output/task2_final.png", final_gray)
print("""Fusion logic: CT carries edges/bone, MRI carries soft-tissue contrast. Weighting CT at 0.7
keeps the boundaries crisp while the 0.3 MRI heatmap tints internal tissue. Higher MRI weight
blurs the edges (see the sweep).""")

In [ ]:
# ===================== TASK 3 — Real-Time Echocardiogram Video Analysis =====================
path = PATHS["echo"]
cap = cv2.VideoCapture(path)                               # file, not webcam (0)
if not cap.isOpened():
    print("Echo video not found -> writing a synthetic clip to", path)
    synthetic_echo(path)
    cap = cv2.VideoCapture(path)

fps = cap.get(cv2.CAP_PROP_FPS) or 25
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter("output/task3_echo_side_by_side.mp4",
                         cv2.VideoWriter_fourcc(*"mp4v"), fps, (W * 2, H))
samples, idx = [], 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:                                            # end of video
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    g = cv2.equalizeHist(gray)                             # 1) fight murky contrast
    g = log_transform(g)                                   # 4) reveal dark chambers
    g = gamma_transform(g, 1.5)                            # 5) gamma>1 suppresses bright backscatter
    heat = cv2.applyColorMap(g, cv2.COLORMAP_JET)          # 2) heatmap of intensities
    enhanced = gray_world(heat)                            # 3) colour balance
    monitor = np.hstack([frame, enhanced])                 # same height & 3 channels
    writer.write(monitor)

    if SHOW_LIVE:
        cv2.imshow("Raw | Enhanced  (q to quit)", monitor)
        if cv2.waitKey(int(1000 / fps)) & 0xFF == ord("q"):
            break
    elif idx % 15 == 0 and len(samples) < 4:
        samples.append(monitor)
    idx += 1

cap.release(); writer.release()
if SHOW_LIVE:
    cv2.destroyAllWindows()
print(f"Processed {idx} frames at {fps:.0f} fps -> output/task3_echo_side_by_side.mp4")
if samples:
    show(samples, [f"Raw | Enhanced (sample {i + 1})" for i in range(len(samples))], cols=2)
# Order note: the task lists heatmap before log/gamma, but log/gamma are gray-level transforms,
# so they are applied to the gray frame first, then mapped to colour and balanced.

In [ ]:
# ===================== EXTRA — Piecewise-linear transforms (named in exam announcement) =====================
# Contrast stretching through (0,0)-(r1,s1)-(r2,s2)-(255,255), min-max stretch,
# intensity-level slicing, thresholding as a special case, and the transfer-function plot.
img = cv2.imread(PATHS["xray"], cv2.IMREAD_GRAYSCALE)
if img is None:
    img = synthetic_xray()


def stretch_lut(r1, s1, r2, s2):
    assert r1 <= r2 and s1 <= s2, "keep the function monotonic"
    return np.interp(np.arange(256), [0, r1, r2, 255], [0, s1, s2, 255]).astype(np.uint8)


rmin, rmax = int(img.min()), int(img.max())
stretch = cv2.LUT(img, stretch_lut(rmin, 0, rmax, 255))           # = min-max stretch
custom = cv2.LUT(img, stretch_lut(20, 10, 70, 240))               # expand the 20..70 band
minmax = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
thresh = cv2.LUT(img, stretch_lut(45, 0, 45, 255))                # r1=r2, s1=0, s2=255 -> threshold
band = ((img >= 30) & (img <= 60)).astype(np.uint8) * 255         # slicing, binary
band_keep = img.copy(); band_keep[(img >= 30) & (img <= 60)] = 255  # slicing, keep background

show([img, stretch, custom, minmax, thresh, band, band_keep],
     ["Original", f"Stretch ({rmin},0)-({rmax},255)", "Stretch (20,10)-(70,240)",
      "cv2.normalize min-max", "r1=r2=45 -> threshold", "Slice [30,60] binary", "Slice [30,60] keep"])

r = np.arange(256)
plt.figure(figsize=(6, 5))
plt.plot(r, stretch_lut(rmin, 0, rmax, 255), label="min-max stretch")
plt.plot(r, stretch_lut(20, 10, 70, 240), label="(20,10)-(70,240)")
plt.plot(r, stretch_lut(45, 0, 45, 255), label="threshold r1=r2")
plt.plot(r, r, "k:", label="identity (r1=s1, r2=s2)")
plt.xlabel("input r"); plt.ylabel("output s"); plt.legend(); plt.title("Piecewise transfer functions"); plt.show()